# 02 — ETL Cleaning Pipeline

Merges all 8 Olist raw tables into two flat, analysis-ready master datasets and exports them to `data/processed/` for Tableau.

| Output File | Purpose |
|---|---|
| `olist_master_dataset.csv` | Full order-level fact table for KPI & delivery analysis |
| `olist_geospatial_summary.csv` | State-level aggregation for the Brazil map visual |

## 1 — Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().resolve().name == 'notebooks'
    else Path.cwd().resolve()
)
RAW  = PROJECT_ROOT / 'data' / 'raw'
PROC = PROJECT_ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {RAW}')
print(f'Processed    : {PROC}')

Project root : /Users/anandmishra1/Downloads/DVA_2/Olist_Ecommerce_Business_Insights_Sec-C_W-8
Raw data     : /Users/anandmishra1/Downloads/DVA_2/Olist_Ecommerce_Business_Insights_Sec-C_W-8/data/raw
Processed    : /Users/anandmishra1/Downloads/DVA_2/Olist_Ecommerce_Business_Insights_Sec-C_W-8/data/processed


## 2 — Load Raw Tables

In [2]:
orders      = pd.read_csv(RAW / 'olist_orders_dataset.csv')
items       = pd.read_csv(RAW / 'olist_order_items_dataset.csv')
customers   = pd.read_csv(RAW / 'olist_customers_dataset.csv')
reviews     = pd.read_csv(RAW / 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(RAW / 'olist_products_dataset.csv')
sellers     = pd.read_csv(RAW / 'olist_sellers_dataset.csv')
translation = pd.read_csv(RAW / 'product_category_name_translation.csv')
geo         = pd.read_csv(RAW / 'olist_geolocation_dataset.csv')

for name, df in [
    ('orders', orders), ('items', items), ('customers', customers),
    ('reviews', reviews), ('products', products), ('sellers', sellers),
    ('translation', translation), ('geo', geo)
]:
    print(f'{name:15s}: {df.shape}')

orders         : (99441, 8)
items          : (112650, 7)
customers      : (99441, 5)
reviews        : (99224, 7)
products       : (32951, 9)
sellers        : (3095, 4)
translation    : (71, 2)
geo            : (1000163, 5)


## 3 — Translate Product Categories (Portuguese → English)

In [3]:
products = pd.merge(products, translation, on='product_category_name', how='left')
products.drop(columns=['product_category_name'], inplace=True)

print('Products with English categories:', products['product_category_name_english'].notna().sum())
products[['product_id', 'product_category_name_english']].head()

Products with English categories: 32328


,product_id,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares


## 4 — Aggregate Items per Order

In [4]:
items_agg = (
    items
    .sort_values('price', ascending=False)
    .groupby('order_id', as_index=False)
    .agg(
        product_id       = ('product_id',    'first'),
        seller_id        = ('seller_id',     'first'),
        order_item_count = ('order_item_id', 'count'),
        total_price      = ('price',         'sum'),
        total_freight    = ('freight_value', 'sum'),
        revenue          = ('price',         'sum'),
    )
)
print('Aggregated items rows:', len(items_agg))
items_agg.head()

Aggregated items rows: 98666


,order_id,product_id,seller_id,order_item_count,total_price,total_freight,revenue
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,1,199.90,18.14,199.90


## 5 — Aggregate Reviews per Order

In [5]:
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')

reviews_agg = (
    reviews
    .sort_values('review_creation_date', ascending=False)
    .groupby('order_id', as_index=False)
    .agg(
        review_score           = ('review_score',           'first'),
        review_comment_message = ('review_comment_message', 'first'),
    )
)
print('Aggregated reviews rows:', len(reviews_agg))
reviews_agg.head()

Aggregated reviews rows: 98673


,order_id,review_score,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,5,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4,NaN
2,000229ec398224ef6ca0657da4fc703e,5,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4,NaN
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,Gostei pois veio no prazo determinado .


## 6 — Build Master Dataset

In [6]:
master_df = (
    orders
    .merge(customers,   on='customer_id', how='left')
    .merge(items_agg,   on='order_id',    how='left')
    .merge(products,    on='product_id',  how='left')
    .merge(sellers,     on='seller_id',   how='left')
    .merge(reviews_agg, on='order_id',    how='left')
)

print(f'Master shape before cleaning: {master_df.shape}')
master_df.head(3)

Master shape before cleaning: (99441, 31)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,review_score,review_comment_message
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,500.0,19.0,8.0,13.0,housewares,9350.0,maua,SP,4.0,"Não testei o produto ainda, mas ele veio corre..."
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,400.0,19.0,13.0,19.0,perfumery,31570.0,belo horizonte,SP,4.0,Muito bom o produto.
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,420.0,24.0,19.0,21.0,auto,14840.0,guariba,SP,5.0,NaN


## 7 — Handle Nulls & Filter Invalid Orders

In [7]:
before = len(master_df)
master_df = master_df[~master_df['order_status'].isin(['canceled', 'unavailable'])].reset_index(drop=True)
print(f'Dropped {before - len(master_df)} rows. Remaining: {len(master_df)}')

master_df['review_comment_message']       = master_df['review_comment_message'].fillna('No Comment')
master_df['product_category_name_english'] = master_df['product_category_name_english'].fillna('Unknown')

Dropped 1234 rows. Remaining: 98207


## 8 — Datetime Conversion

In [8]:
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'order_approved_at',
    'order_delivered_carrier_date',
]
for col in date_cols:
    if col in master_df.columns:
        master_df[col] = pd.to_datetime(master_df[col], errors='coerce')

master_df.dtypes[date_cols]

order_purchase_timestamp         datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
dtype: object

## 9 — Feature Engineering

In [9]:
master_df['delivery_time_days']  = (master_df['order_delivered_customer_date'] - master_df['order_purchase_timestamp']).dt.days
master_df['delivery_delay_days'] = (master_df['order_delivered_customer_date'] - master_df['order_estimated_delivery_date']).dt.days
master_df['is_late']             = master_df['delivery_delay_days'] > 0
master_df['order_year_month']    = master_df['order_purchase_timestamp'].dt.to_period('M').astype(str)
master_df['order_year']          = master_df['order_purchase_timestamp'].dt.year

master_df[[
    'order_id', 'order_purchase_timestamp',
    'order_delivered_customer_date', 'order_estimated_delivery_date',
    'delivery_time_days', 'delivery_delay_days', 'is_late'
]].dropna(subset=['delivery_delay_days']).head(5)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_time_days,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,False


## 10 — Data Dictionary

| Column Name | Data Type | Description | Example |
|---|---|---|---|
| `delivery_time_days` | int | Days from purchase to actual delivery | 12 |
| `delivery_delay_days` | int | Actual minus estimated delivery (positive = late, negative = early) | 3 |
| `is_late` | bool | True if order arrived after the estimated date | True |
| `order_year_month` | str | Year-month period for time-series grouping | 2018-01 |
| `order_year` | int | Calendar year of purchase | 2018 |
| `order_item_count` | int | Number of line items in the order | 2 |
| `total_price` | float | Sum of item prices | 89.90 |
| `total_freight` | float | Sum of freight charges | 14.50 |
| `revenue` | float | Alias of total_price for Tableau | 89.90 |

## 11 — Export: olist_master_dataset.csv

In [10]:
MASTER_OUT = PROC / 'olist_master_dataset.csv'
master_df.to_csv(MASTER_OUT, index=False)
print(f'Saved: {MASTER_OUT}')
print(f'Rows: {len(master_df):,}  |  Columns: {len(master_df.columns)}')

Saved: /Users/anandmishra1/Downloads/DVA_2/Olist_Ecommerce_Business_Insights_Sec-C_W-8/data/processed/olist_master_dataset.csv
Rows: 98,207  |  Columns: 36


## 12 — Export: olist_geospatial_summary.csv

In [11]:
geo_summary = (
    master_df
    .groupby('customer_state', as_index=False)
    .agg(
        total_orders            = ('order_id',             'count'),
        avg_delivery_time_days  = ('delivery_time_days',   'mean'),
        avg_delivery_delay_days = ('delivery_delay_days',  'mean'),
        late_orders             = ('is_late',              'sum'),
        total_revenue           = ('revenue',              'sum'),
        avg_review_score        = ('review_score',         'mean'),
    )
)
geo_summary['on_time_rate'] = (
    (geo_summary['total_orders'] - geo_summary['late_orders']) / geo_summary['total_orders']
).round(4)

for col in ['avg_delivery_time_days', 'avg_delivery_delay_days', 'avg_review_score', 'total_revenue']:
    geo_summary[col] = geo_summary[col].round(2)

GEO_OUT = PROC / 'olist_geospatial_summary.csv'
geo_summary.to_csv(GEO_OUT, index=False)
print(f'Saved: {GEO_OUT}')
print(f'Rows: {len(geo_summary)} (one per state)')
geo_summary.sort_values('avg_delivery_delay_days', ascending=False).head(10)

Saved: /Users/anandmishra1/Downloads/DVA_2/Olist_Ecommerce_Business_Insights_Sec-C_W-8/data/processed/olist_geospatial_summary.csv
Rows: 27 (one per state)


,customer_state,total_orders,avg_delivery_time_days,avg_delivery_delay_days,late_orders,total_revenue,avg_review_score,on_time_rate
1,AL,411,24.04,-8.71,85,80314.81,3.76,0.7932
9,MA,736,21.12,-9.57,125,119291.62,3.78,0.8302
24,SE,345,21.03,-10.02,51,58920.85,3.84,0.8522
7,ES,2018,15.33,-10.50,214,273532.13,4.05,0.8940
4,BA,3344,18.87,-10.79,396,507108.83,3.88,0.8816
5,CE,1323,20.82,-10.80,176,226264.06,3.88,0.8670
11,MS,708,15.19,-11.05,68,116754.65,4.14,0.9040
25,SP,41127,8.30,-11.08,1820,5163867.22,4.21,0.9557
16,PI,490,18.99,-11.31,66,86660.09,3.93,0.8653
23,SC,3600,14.48,-11.50,291,518578.28,4.10,0.9192


## 13 — Final Validation

In [12]:
print(f'Total orders     : {len(master_df):,}')
print(f'Unique customers : {master_df["customer_unique_id"].nunique():,}')
print(f'Order statuses   : {master_df["order_status"].value_counts().to_dict()}')
print(f'Date range       : {master_df["order_purchase_timestamp"].min().date()} → {master_df["order_purchase_timestamp"].max().date()}')
print(f'Nulls in is_late : {master_df["is_late"].isna().sum()}')

Total orders     : 98,207
Unique customers : 94,990
Order statuses   : {'delivered': 96478, 'shipped': 1107, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}
Date range       : 2016-09-04 → 2018-09-03
Nulls in is_late : 0
